# 01 — Data Preparation

**Project:** Post-Only (A) vs Trajectory (B) Supervision for Continual Tool-Use Learning

This notebook:
1. Downloads API-Bank from HuggingFace
2. Extracts API names, excludes ToolSearcher (meta-API)
3. Builds 6 balanced domain blocks via greedy bin-packing
4. Creates 80/20 train/eval splits per block
5. Formats data for Conditions A, B, and A+ (Mistral instruct format)
6. Saves preprocessed data as pickle

In [ ]:
!pip install -q transformers datasets huggingface_hub numpy tqdm

In [ ]:
import json
import os
import re
import random
import pickle
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Seed: {SEED}")

## 1. Download API-Bank

In [ ]:
data_files = [
    "training-data/lv1-train.json",
    "training-data/lv2-train.json",
    "training-data/lv3-train.json",
]

all_raw = []
for fname in data_files:
    path = hf_hub_download(
        repo_id="liminghao1630/API-Bank",
        filename=fname,
        repo_type="dataset",
    )
    with open(path) as f:
        entries = json.load(f)
    print(f"{fname}: {len(entries)} entries")
    all_raw.extend(entries)

print(f"\nTotal raw entries: {len(all_raw)}")
print(f"Keys: {list(all_raw[0].keys())}")

## 2. Extract API Names & Filter

In [ ]:
def extract_api_name(entry):
    # Extract primary API name from entry output or input
    text = entry.get('output', '') or ''
    match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', text)
    if match:
        return match.group(1)
    text = entry.get('input', '') or ''
    match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', text)
    if match:
        return match.group(1)
    return 'unknown'

for entry in all_raw:
    entry['api_name'] = extract_api_name(entry)

api_counts = defaultdict(int)
for entry in all_raw:
    api_counts[entry['api_name']] += 1

sorted_apis = sorted(api_counts.items(), key=lambda x: -x[1])
print(f"Unique APIs: {len(sorted_apis)}")
print(f"ToolSearcher entries: {api_counts.get('ToolSearcher', 0)}")
print(f"\nTop 10 APIs:")
for api, count in sorted_apis[:10]:
    print(f"  {api}: {count}")

In [ ]:
# Exclude ToolSearcher (meta-API) and unknown
filtered = [e for e in all_raw if e['api_name'] not in ('ToolSearcher', 'unknown')]
print(f"After excluding ToolSearcher: {len(filtered)} entries "
      f"(removed {len(all_raw) - len(filtered)})")

MIN_ENTRIES = 10
api_counts_filtered = defaultdict(int)
for entry in filtered:
    api_counts_filtered[entry['api_name']] += 1

valid_apis = sorted(
    [api for api, count in api_counts_filtered.items() if count >= MIN_ENTRIES]
)
valid_entries = [e for e in filtered if e['api_name'] in valid_apis]

print(f"APIs with >= {MIN_ENTRIES} entries: {len(valid_apis)}")
print(f"Entries from valid APIs: {len(valid_entries)}")

## 3. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")

## 4. Format Functions (Mistral Instruct Format)

Both conditions use the **same system prompt** and **same output**.
Only difference: whether input context includes prior API interactions.

- **Condition A (Post-Only):** API-Request/Response lines removed from input
- **Condition B (Trajectory):** Full dialogue kept in input

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that can use tools. "
    "When you need to call an API, use the format: "
    "[ApiName(param1='value1', param2='value2')]. "
    "After receiving the API response, use it to formulate your answer."
)

def _strip_api_lines(text):
    # Remove API-Request, API-Response, and related lines
    lines = text.split('\n')
    out = []
    for line in lines:
        s = line.strip()
        if s.startswith('API-Request:') or s.startswith('API-Response:'):
            continue
        if 'Received API Response' in line or 'Generate API Request' in line:
            continue
        out.append(line)
    return '\n'.join(out).strip()

def format_entry(entry, condition, tokenizer):
    # Format a single entry. Returns (full_text, prompt_token_len).
    inp = entry['input']
    out = entry.get('output', '')

    if condition == 'A':
        context = _strip_api_lines(inp)
    else:
        context = inp

    prompt = f"[INST] {SYSTEM_PROMPT}\n\n{context} [/INST]"
    response = f" {out}</s>"
    full_text = prompt + response
    prompt_len = len(tokenizer.encode(prompt, add_special_tokens=False))
    return full_text, prompt_len

# Quick test
sample = valid_entries[0]
for cond in ['A', 'B']:
    text, plen = format_entry(sample, cond, tokenizer)
    total = len(tokenizer.encode(text))
    print(f"Condition {cond}: {total} tokens (prompt: {plen}, response: {total - plen})")
print(f"\nSample output: {sample.get('output', '')[:100]}")

## 5. Compute Token Counts & Greedy Bin-Packing

In [ ]:
# Estimate token counts per API
api_stats = {}
for api in tqdm(valid_apis, desc="Token counting"):
    entries = [e for e in valid_entries if e['api_name'] == api]
    sample = entries[:min(20, len(entries))]
    tokens_a = [len(tokenizer.encode(format_entry(e, 'A', tokenizer)[0])) for e in sample]
    tokens_b = [len(tokenizer.encode(format_entry(e, 'B', tokenizer)[0])) for e in sample]
    api_stats[api] = {
        'count': len(entries),
        'avg_tokens_a': np.mean(tokens_a),
        'avg_tokens_b': np.mean(tokens_b),
        'est_total_b': np.mean(tokens_b) * len(entries),
    }

total_entries = sum(s['count'] for s in api_stats.values())
total_tokens_b = sum(s['est_total_b'] for s in api_stats.values())
print(f"Total entries: {total_entries}")
print(f"Estimated total tokens (B): {total_tokens_b:,.0f}")

In [ ]:
# Greedy bin-packing into 6 blocks
NUM_BLOCKS = 6
sorted_api_names = sorted(valid_apis, key=lambda x: -api_stats[x]['est_total_b'])

block_api_lists = [[] for _ in range(NUM_BLOCKS)]
block_token_totals = [0.0] * NUM_BLOCKS

for api in sorted_api_names:
    min_idx = int(np.argmin(block_token_totals))
    block_api_lists[min_idx].append(api)
    block_token_totals[min_idx] += api_stats[api]['est_total_b']

print("Block composition:")
for i in range(NUM_BLOCKS):
    n_apis = len(block_api_lists[i])
    n_entries = sum(api_stats[a]['count'] for a in block_api_lists[i])
    print(f"  D{i+1}: {n_apis} APIs, {n_entries} entries, ~{block_token_totals[i]:,.0f} tokens")

## 6. Build Domain Blocks with Train/Eval Splits

In [ ]:
MAX_SEQ_LEN = 2048

domain_blocks = []
for i in range(NUM_BLOCKS):
    block_entries = [e for e in valid_entries if e['api_name'] in block_api_lists[i]]
    random.shuffle(block_entries)
    split_idx = int(len(block_entries) * 0.8)
    train_entries = block_entries[:split_idx]
    eval_entries = block_entries[split_idx:]
    domain_blocks.append({
        'block_id': i + 1,
        'apis': block_api_lists[i],
        'train_entries': train_entries,
        'eval_entries': eval_entries,
    })
    print(f"D{i+1}: {len(train_entries)} train, {len(eval_entries)} eval")

## 7. Format All Data & Compute Token Stats

In [ ]:
blocks_data = []
BASE_EPOCHS = 3

for block in tqdm(domain_blocks, desc="Formatting"):
    bid = block['block_id']
    train_a, train_a_plens = [], []
    train_b, train_b_plens = [], []
    for e in block['train_entries']:
        ta, pa = format_entry(e, 'A', tokenizer)
        tb, pb = format_entry(e, 'B', tokenizer)
        train_a.append(ta); train_a_plens.append(pa)
        train_b.append(tb); train_b_plens.append(pb)

    eval_a, eval_a_plens = [], []
    eval_b, eval_b_plens = [], []
    for e in block['eval_entries']:
        ta, pa = format_entry(e, 'A', tokenizer)
        tb, pb = format_entry(e, 'B', tokenizer)
        eval_a.append(ta); eval_a_plens.append(pa)
        eval_b.append(tb); eval_b_plens.append(pb)

    tokens_a = sum(min(len(tokenizer.encode(t)), MAX_SEQ_LEN) for t in train_a)
    tokens_b = sum(min(len(tokenizer.encode(t)), MAX_SEQ_LEN) for t in train_b)
    ratio = tokens_b / tokens_a if tokens_a > 0 else 1.0
    aplus_epochs = max(round(BASE_EPOCHS * ratio), BASE_EPOCHS)

    blocks_data.append({
        'block_id': bid,
        'apis': block['apis'],
        'train_a': train_a, 'train_b': train_b,
        'train_a_prompt_lens': train_a_plens, 'train_b_prompt_lens': train_b_plens,
        'eval_a': eval_a, 'eval_b': eval_b,
        'eval_a_prompt_lens': eval_a_plens, 'eval_b_prompt_lens': eval_b_plens,
        'eval_entries_raw': block['eval_entries'],
        'train_tokens_a': tokens_a, 'train_tokens_b': tokens_b,
        'token_ratio': ratio, 'aplus_epochs': aplus_epochs,
    })
    print(f"D{bid}: A={tokens_a:,} tok, B={tokens_b:,} tok, "
          f"ratio={ratio:.2f}x, A+ epochs={aplus_epochs}")

## 8. Save Preprocessed Data

In [ ]:
OUTPUT_DIR = "preprocessed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_data = {
    'blocks': blocks_data,
    'config': {
        'model_name': MODEL_NAME,
        'num_blocks': NUM_BLOCKS,
        'max_seq_len': MAX_SEQ_LEN,
        'base_epochs': BASE_EPOCHS,
        'seed': SEED,
        'min_entries': MIN_ENTRIES,
        'toolsearcher_excluded': True,
        'total_valid_apis': len(valid_apis),
        'total_valid_entries': len(valid_entries),
        'system_prompt': SYSTEM_PROMPT,
    },
}

pkl_path = os.path.join(OUTPUT_DIR, 'preprocessed.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(save_data, f)

# Human-readable summary
summary = {
    'config': save_data['config'],
    'blocks': [{
        'block_id': b['block_id'],
        'num_apis': len(b['apis']),
        'apis': b['apis'][:10],
        'num_train': len(b['train_a']),
        'num_eval': len(b['eval_a']),
        'train_tokens_a': b['train_tokens_a'],
        'train_tokens_b': b['train_tokens_b'],
        'token_ratio': round(b['token_ratio'], 3),
        'aplus_epochs': b['aplus_epochs'],
    } for b in blocks_data],
}
json_path = os.path.join(OUTPUT_DIR, 'summary.json')
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)

pkl_size = os.path.getsize(pkl_path) / 1e6
print(f"\nSaved to {OUTPUT_DIR}/")
print(f"  preprocessed.pkl  ({pkl_size:.1f} MB)")
print(f"  summary.json")